# Tokopedia Web Scraping

In [ ]:
import nbformat

nb = nbformat.read("/content/drive/MyDrive/Colab Notebooks/Project6_Tokopedia Web Scraping and Sentiment Analisis.ipynb", as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, "Project6_Tokopedia Web Scraping and Sentiment Analisis.ipynb")

In [1]:
from google.colab import drive
drive.mount('/content/drive') # Mount Google Drive to access files

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
! pip install selenium # Install Selenium library for web automation
! pip install webdriver_manager # Install webdriver_manager to automatically manage browser drivers
! pip install undetected_chromedriver # Install undetected_chromedriver for scraping websites that detect regular WebDriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for undetected_chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47047 sha256=6ae1857026b2dbfb191c3b0d270c86c7f2f40501701050235e8930b30f4a925a
  Stored in directory: /root/.cache/pip/wheels/c4/f1/aa/9de6cf276210554d91e9c0526864563e850a428c5e76da4914
Successfully built undetected_chromedriver


In [13]:
from selenium import webdriver # Import webdriver from selenium
from selenium.webdriver.chrome.service import Service # Import Service to manage Chrome driver service
from webdriver_manager.chrome import ChromeDriverManager # Import ChromeDriverManager to install/manage Chrome driver
import time # Import time for delays
import undetected_chromedriver as uc # Import undetected_chromedriver for advanced scraping
from selenium.webdriver.common.by import By # Import By for locating elements
from selenium.webdriver.support.ui import WebDriverWait # Import WebDriverWait for explicit waits
from selenium.webdriver.support import expected_conditions as EC # Import expected_conditions for wait conditions

In [14]:
# Install Google Chrome Stable and its dependencies
!apt-get update
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable

# Setup WebDriver
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.binary_location = '/usr/bin/google-chrome'

# Initialize the Chrome driver, allowing ChromeDriverManager to find the correct version automatically
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

print("WebDriver initialized successfully in headless mode.")

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,744 kB]
Get:13 https://cli.github.com/packages stable/main am

In [15]:
import requests # Import requests library for making HTTP requests


def converted_shortlink(short_url):
  # Check if the URL is a Tokopedia shortlink
  if "tk.tokopedia.com" in short_url:
    # Define headers to mimic a web browser
    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

    # Make a GET request to the shortlink without following redirects
    res = requests.get(
      short_url,
      headers=headers,
      allow_redirects=False,
      timeout=10
)
    # Return the 'Location' header which contains the full URL
    return res.headers.get("Location")
  else:
    # If it's not a Tokopedia shortlink, return the original URL
    return short_url

In [16]:
import undetected_chromedriver as uc # Import undetected_chromedriver for advanced scraping
from selenium.webdriver.common.by import By # Import By for locating elements
import time # Import time for delays

def scraping_reviews(url):
    options = uc.ChromeOptions() # Create ChromeOptions object
    options.add_argument('--headless') # Run Chrome in headless mode (without a GUI)
    options.add_argument('--no-sandbox') # Bypass OS security model
    options.add_argument('--disable-dev-shm-usage') # Overcome limited resource problems
    # Tambahkan User-Agent agar lebih mirip manusia
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

    driver = uc.Chrome(options=options) # Initialize undetected_chromedriver
    driver.get(url) # Navigate to the given URL
    driver.implicitly_wait(10) # Set implicit wait for elements to be found

    reviews_data = [] # Initialize list to store review data


    # Get product name
    product_name = driver.find_element(By.CSS_SELECTOR, "h1[data-testid='lblPDPDetailProductName']")
    product_name = product_name.text # Extract text of product name
    print(f"Product Name: {product_name}") # Print product name

    try:
        # Scroll to make reviews visible (required on Tokopedia)
        driver.execute_script("window.scrollTo(0, 2000);")
        time.sleep(5) # Wait for content to load after scrolling

        # 1. Find ALL review containers (using 'article' tag or a wrapper class)
        # If data-testid 'unify-review-card' is not present, use the 'article' tag
        review_box = driver.find_elements(By.TAG_NAME, "article")

        print(f"Find {len(review_box)} review box(es).\n") # Print number of reviews found

        # 2. Iterate/Loop through each review box
        for box in review_box:
            try:
                # Get Name (search only within this 'box')
                name = box.find_element(By.CSS_SELECTOR, "span[class*='name']").text

                # Get Review Text
                review = box.find_element(By.CSS_SELECTOR, "span[data-testid='lblItemUlasan']").text

                # Get Stars from aria-label
                # Use a small try-except here in case a review has no stars
                try:
                    star_el = box.find_element(By.CSS_SELECTOR, "div[data-testid='icnStarRating']")
                    rating = star_el.get_attribute("aria-label") # Result: "bintang 5"
                except:
                    rating = "No Rating"

                # Add to list as a dictionary
                reviews_data.append({
                    "Name": name,
                    "Rating": rating,
                    "Review": review
                })

            except Exception as e:
                # If one review fails (e.g., an ad in between reviews), skip to the next review
                continue

    finally:
        driver.quit() # Close the browser when done or if an error occurs

    # Prepare the final JSON output structure
    hasil_json = {
    "Product": product_name,
    "Review": reviews_data
    }

    return hasil_json

In [17]:
# Convert the shortlink to a full URL
url = converted_shortlink("https://tk.tokopedia.com/ZSQqvqcxu/")
# Scrape all reviews from the URL
results = scraping_reviews(url)

Product Name: ACNAWAY Mugwort Gel Facial Wash Cleanser Acne Friendly Face Wash - Sabun Cuci Muka untuk Kulit Berjerawat & Sensitif Membersihkan Kotoran & Mencerahkan with Mugwort + Centella + Panthenol - Face Wash
Find 11 review box(es).



In [18]:
import json # Import the json library for working with JSON data

# Open a JSON file
with open("tokopedia_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Successfully saved into JSON!") # Confirm that the file has been saved

Successfully saved into JSON!


In [19]:
json_text = json.dumps(results, indent=2, ensure_ascii=False) # Convert the 'results' dictionary to a formatted JSON string
print(json_text) # Print the JSON string

{
  "Product": "ACNAWAY Mugwort Gel Facial Wash Cleanser Acne Friendly Face Wash - Sabun Cuci Muka untuk Kulit Berjerawat & Sensitif Membersihkan Kotoran & Mencerahkan with Mugwort + Centella + Panthenol - Face Wash",
  "Review": [
    {
      "Name": "H***l",
      "Rating": "bintang 5",
      "Review": "Packingnya aman bgt..tebal bgt bubblewrapnya.. Wanginya soft enak.smoga cocok buat anakku Bahan: Mugwort dan centella Manfaat produk: Membersihkan muka Jenis kulit: Bermi..."
    },
    {
      "Name": "Y***o",
      "Rating": "bintang 5",
      "Review": "Pengiriman ny gercep,packing aman,buble wrape tebel bgt,nggak ada kebocoran,kekurangan. Produk ny baguss,cocok buat kulit sensitif"
    },
    {
      "Name": "U***h",
      "Rating": "bintang 5",
      "Review": "Mantul banget face wash nya, after nya bikin kulit ga ketarik, cocok buat aku yang acne prone skin dan beruntusan dikit, semoga cocok buat kedepannya dan bisa repurchase,..."
    },
    {
      "Name": "B***i",
      "Rati

# Using Fine-Tuned Model for Sentiment Analysis

In [8]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

model_name = "raissulaiman/indobert-tokopedia-sentiment"

tokenizer = AutoTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
id2label = {0: "Negative", 1: "Positive"} # Map label IDs to sentiment strings
label2id = {"Negative": 0, "Positive": 1} # Map sentiment strings to label IDs



# Predicting example

In [10]:
from transformers import AutoTokenizer
import torch.nn.functional as F
import torch


#review example
review = "packing tidak aman, diterima dalam kondisi penyet"

print(f"Review: {review}")
# Set the model to evaluation mode
model.eval()
# Determine the device to use (GPU if available, otherwise CPU)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# Move the model to the selected device
model.to(device)

# Tokenize the input review: add padding, truncate to max_length, and return PyTorch tensors
inputs = tokenizer(review, padding=True, truncation=True, max_length=128, return_tensors='pt')
# Move the tokenized inputs to the selected device
inputs.to(device)

# Make a prediction using the model
logits = model(**inputs).logits
# Get the predicted label (index of the highest logit) for each input
labels = [torch.topk(logit, k=1, dim=-1)[1].squeeze().item() for logit in logits]
# Print the predicted sentiment label and its confidence score
print(f"{id2label[labels[0]]} ({F.softmax(logits[0], dim=-1).squeeze()[labels[0]] * 100:.3f}%)")

Review: packing tidak aman, diterima dalam kondisi penyet
Negative (99.897%)


# Data from scraping

In [20]:
import pandas as pd # Import pandas for data manipulation and analysis

df = pd.DataFrame(results["Review"]) # Create a pandas DataFrame from the 'ulasan' list in 'hasil'

# Add the product name as a new column to the DataFrame
df["Product"] = results["Product"]
df # Display the DataFrame

,Name,Rating,Review,Product
0,H***l,bintang 5,Packingnya aman bgt..tebal bgt bubblewrapnya.....,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
1,Y***o,bintang 5,"Pengiriman ny gercep,packing aman,buble wrape ...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
2,U***h,bintang 5,"Mantul banget face wash nya, after nya bikin k...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
3,B***i,bintang 5,Barang udah sampai dan kondisinya baik Bahan: ...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
4,J***w,bintang 5,"beli yg ke 2, aku dpt rekomendasi dari temen d...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
5,M***z,bintang 5,Maaf Bru ngulas soalnya mau nyoba dulu Alhamdu...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
6,D***r,bintang 5,Packaging nya cakep rapih foll seriusan dh coc...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
7,T***h,bintang 5,"bagus,cocok untuk semua tipkul,rekomen buat ti...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
8,H***e,bintang 5,Beli untuk ke 2 kalinya karena pada saat itu b...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...
9,C***o,bintang 5,"Order untuk yg ksekian kalinya , biasanya beli...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...


In [21]:
for i in df['Review']:
  model.eval()
  device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
  model.to(device)

  inputs = tokenizer(i, padding=True, truncation=True, max_length=512, return_tensors='pt')
  inputs.to(device)

  # Make a prediction using the model
  logits = model(**inputs).logits
  # Get the predicted label (index of the highest logit) for each input
  labels = [torch.topk(logit, k=1, dim=-1)[1].squeeze().item() for logit in logits]
  # Add the predicted sentiment label to the DataFrame for the current review
  df.loc[df['Review'] == i, 'Sentiment'] = id2label[labels[0]]

df

,Name,Rating,Review,Product,Sentiment
0,H***l,bintang 5,Packingnya aman bgt..tebal bgt bubblewrapnya.....,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Positive
1,Y***o,bintang 5,"Pengiriman ny gercep,packing aman,buble wrape ...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Positive
2,U***h,bintang 5,"Mantul banget face wash nya, after nya bikin k...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Positive
3,B***i,bintang 5,Barang udah sampai dan kondisinya baik Bahan: ...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Positive
4,J***w,bintang 5,"beli yg ke 2, aku dpt rekomendasi dari temen d...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Negative
5,M***z,bintang 5,Maaf Bru ngulas soalnya mau nyoba dulu Alhamdu...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Negative
6,D***r,bintang 5,Packaging nya cakep rapih foll seriusan dh coc...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Positive
7,T***h,bintang 5,"bagus,cocok untuk semua tipkul,rekomen buat ti...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Positive
8,H***e,bintang 5,Beli untuk ke 2 kalinya karena pada saat itu b...,ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Negative
9,C***o,bintang 5,"Order untuk yg ksekian kalinya , biasanya beli...",ACNAWAY Mugwort Gel Facial Wash Cleanser Acne ...,Negative


In [22]:
json_nested = {
    "Product": df["Product"].iloc[0], # Get the product name from the first row of the 'produk' column
    "Review": df.drop(columns="Product").to_dict(orient="records") # Convert the DataFrame (excluding 'produk' column) to a list of dictionaries
}

In [23]:
print(json_nested) # Print the nested JSON structure

{'Product': 'ACNAWAY Mugwort Gel Facial Wash Cleanser Acne Friendly Face Wash - Sabun Cuci Muka untuk Kulit Berjerawat & Sensitif Membersihkan Kotoran & Mencerahkan with Mugwort + Centella + Panthenol - Face Wash', 'Review': [{'Name': 'H***l', 'Rating': 'bintang 5', 'Review': 'Packingnya aman bgt..tebal bgt bubblewrapnya.. Wanginya soft enak.smoga cocok buat anakku Bahan: Mugwort dan centella Manfaat produk: Membersihkan muka Jenis kulit: Bermi...', 'Sentiment': 'Positive'}, {'Name': 'Y***o', 'Rating': 'bintang 5', 'Review': 'Pengiriman ny gercep,packing aman,buble wrape tebel bgt,nggak ada kebocoran,kekurangan. Produk ny baguss,cocok buat kulit sensitif', 'Sentiment': 'Positive'}, {'Name': 'U***h', 'Rating': 'bintang 5', 'Review': 'Mantul banget face wash nya, after nya bikin kulit ga ketarik, cocok buat aku yang acne prone skin dan beruntusan dikit, semoga cocok buat kedepannya dan bisa repurchase,...', 'Sentiment': 'Positive'}, {'Name': 'B***i', 'Rating': 'bintang 5', 'Review': 'Bar